In [ ]:
"""Your pipeline uses a filtered transactions DataFrame three times: once for merchant-level revenue aggregations, once for customer-level
   aggregations, and once for regional trend analysis. A colleague says adding .cache() will fix the repeated recomputation."""

#==>Before — Wrong Caching (Full Raw Table)
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("Scenario_5")\
    .config("spark.sql.shuffle.partitions", "200") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
#raw_dataframe
df_raw = spark.read.option("mergeSchema","True").parquet(r"D:\data engineer\pyspark_practice\pyspark_scenarios\dataset1_transactions\merchant_lookup")
df_raw.cache()
df_raw.count()  # forces materialisation — triggers spill silently

"""The raw transactions table is 10 million rows. In Spark executor memory, a 10M row DataFrame with 8 columns occupies roughly 3-5GB 
depending on string column cardinality. With spark.memory.fraction=0.4, your executor's Spark memory pool is ~1.5GB. The 3-5GB DataFrame cannot fit.
It partially loads into memory and the rest spills silently to disk. Every time a downstream step needs a row that landed in the spill file, 
Spark reads from disk — 100x slower than memory. Job with wrong cache: 8+ minutes. Job without any cache: ~3 minutes. The cache made it slower."""



"The raw transactions table is 10 million rows. In Spark executor memory, a 10M row DataFrame with 8 columns occupies roughly 3-5GB \ndepending on string column cardinality. With spark.memory.fraction=0.4, your executor's Spark memory pool is ~1.5GB. The 3-5GB DataFrame cannot fit.\nIt partially loads into memory and the rest spills silently to disk. Every time a downstream step needs a row that landed in the spill file, \nSpark reads from disk — 100x slower than memory. Job with wrong cache: 8+ minutes. Job without any cache: ~3 minutes. The cache made it slower."

In [ ]:
#==>After — Correct Selective Caching
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder \
    .appName("Scenario_5")\
    .config("spark.sql.shuffle.partitions", "200") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
#raw_dataframe
df_raw = spark.read.option("mergeSchema","True").parquet(r"D:\data engineer\pyspark_practice\pyspark_scenarios\dataset1_transactions\merchant_lookup")

filter_df=df_raw.filter(col("tier")=="Gold")
filter_df.cache()
filter_df.count()

# Now reuse df_filtered 3 times without recomputation
filter_df.groupBy('merchant_id').agg(count('*')).show()
filter_df.groupBy('merchant_name').agg(count('*')).show()
filter_df.groupBy('category').agg(count('*')).show()


"""failed + pending rows are approximately 21% of 10 million = 2.1 million rows. Selecting only 5 needed columns further reduces the in-memory 
   footprint to approximately 400-600MB. This comfortably fits in the 1.5GB Spark memory pool. Zero spill. All 3 reuses are instant memory reads."""



+-----------+--------+
|merchant_id|count(1)|
+-----------+--------+
|     M00080|       1|
|     M00354|       1|
|     M00110|       1|
|     M00033|       1|
|     M00403|       1|
|     M00395|       1|
|     M00013|       1|
|     M00486|       1|
|     M00257|       1|
|     M00090|       1|
|     M00227|       1|
|     M00472|       1|
|     M00404|       1|
|     M00233|       1|
|     M00291|       1|
|     M00435|       1|
|     M00046|       1|
|     M00297|       1|
|     M00334|       1|
|     M00437|       1|
+-----------+--------+
only showing top 20 rows

+---------------+--------+
|  merchant_name|count(1)|
+---------------+--------+
|Merchant_M00101|       1|
|Merchant_M00392|       1|
|Merchant_M00195|       1|
|Merchant_M00036|       1|
|Merchant_M00021|       1|
|Merchant_M00066|       1|
|Merchant_M00211|       1|
|Merchant_M00166|       1|
|Merchant_M00297|       1|
|Merchant_M00212|       1|
|Merchant_M00110|       1|
|Merchant_M00013|       1|
|Merchant_M00057|